# CatBoost CPU vs GPU Benchmark Notebook

這份 Notebook 用來做 **CatBoost Regressor 在 CPU 與 GPU 上的訓練速度對比實驗**。

## 這份 Notebook 的設計重點

### 1. 實驗目的
你要回答的問題不是單純「GPU kernel 快不快」，而是：

- 在不同 `sample 數量` 與 `feature 數量` 下，GPU 相對 CPU 是否真的變快
- 變快多少
- 穩不穩定
- 哪些規模下 GPU 值得使用
- 哪些規模下 CPU 反而更划算

### 2. 計時定義
這份 Notebook 的 benchmark 定義如下：

- **資料生成時間不納入 benchmark**
- 但是 **CatBoost `.fit()` 內發生的所有成本都納入**
- 也就是說，若 GPU 路徑中包含：
  - 初始化成本
  - device setup
  - data transfer / internal conversion
  - framework overhead

  這些都算進去

這是因為你的目標是看 **從 CPU 換到 GPU 之後，真實 end-to-end fitting latency 有沒有下降**。

### 3. 輸入格式
- 使用 `sklearn.datasets.make_regression` 產生資料
- 將 `X` 轉成 `pandas DataFrame`
- 將 `y` 轉成 `pandas Series`
- CPU 與 GPU 都統一吃 DataFrame / Series

### 4. 實驗方式
- 你提供：
  - `SAMPLE_LIST`
  - `FEATURE_LIST`
  - `N_RUNS`
- 使用 Cartesian product 建立 benchmark grid
- 每個 scenario 跑多次
- 在同一個 `(n_samples, n_features, run_id)` 下，CPU 與 GPU 使用**同一份資料**
- 這樣 speedup 比較比較公平

### 5. 輸出結果
最後會產生：

- 原始實驗結果表 `df_raw`
- 統計摘要表 `summary_df`
- 報告表 `report_table`
- 幾張主要圖表：
  - Speedup heatmap
  - Absolute time saved heatmap
  - 固定 feature 看 time vs samples
  - 固定 sample 看 speedup vs features
  - Break-even map

---

## 你最需要改的地方
最常修改的是下面幾個區塊：

1. **Global Experiment Config**
   - `SAMPLE_LIST`
   - `FEATURE_LIST`
   - `N_RUNS`

2. **CatBoost Global Params**
   - `iterations`
   - `depth`
   - `learning_rate`

3. **CPU / GPU 專屬設定**
   - CPU: `thread_count`
   - GPU: `devices`, `gpu_ram_part`, `pinned_memory_size`

---

## 使用前提醒
若你要跑 GPU 版本，請先確認：

- 你的環境有支援 GPU 的 CatBoost
- NVIDIA driver / CUDA 環境正常
- `task_type="GPU"` 可以正常執行

如果 GPU 環境沒有設定好，GPU 那部分會報錯。


In [ ]:
# =========================
# Cell 1 — Imports
# =========================

import time
import json
import itertools
import warnings
from dataclasses import dataclass
from typing import Optional, Dict, Any, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")


In [ ]:
# =========================
# Cell 2 — 全域設定區
# =========================
# 這一格是整份實驗最常改的地方
# 你通常只要調整這裡，就可以改整體 benchmark 的設定

# 隨機種子：用來確保資料生成與切分比較可重現
RANDOM_SEED = 42

# -------------------------
# Benchmark grid
# -------------------------
# 你要測試的 sample 數量清單
SAMPLE_LIST = [10_000, 50_000, 100_000]

# 你要測試的 feature 數量清單
FEATURE_LIST = [50, 200, 1000, 3000]

# 每個 scenario 要重複跑幾次
# 例如 N_RUNS = 5 表示：
# 每一個 (n_samples, n_features) 會各跑 5 次 CPU + 5 次 GPU
N_RUNS = 5

# train / valid split 比例
TEST_SIZE = 0.2

# -------------------------
# make_regression 資料生成設定
# -------------------------
# informative 特徵占比
# 例如 0.6 表示 n_features 的 60% 會是 informative
N_INFORMATIVE_RATIO = 0.6

# label noise
NOISE = 0.1

# bias 項
BIAS = 0.0

# 若不想指定低秩結構，可維持 None
EFFECTIVE_RANK = None

# 與有效秩結構有關的尾端強度
TAIL_STRENGTH = 0.5

# 是否將資料轉成 float32
# 建議 True，因為：
# 1. 可降低記憶體壓力
# 2. CPU / GPU 輸入格式更一致
USE_FLOAT32 = True

# 畫圖時是否使用對數軸
USE_LOG_X = True
USE_LOG_Y = False

# -------------------------
# CatBoost 通用參數
# -------------------------
# 這些參數 CPU / GPU 共同使用
CATBOOST_COMMON_PARAMS = {
    "iterations": 300,
    "depth": 6,
    "learning_rate": 0.1,
    "loss_function": "RMSE",
    "eval_metric": "RMSE",
    "verbose": False,
    "random_seed": RANDOM_SEED,

    # 你也可以在這裡額外補充參數，例如：
    # "bootstrap_type": "Bernoulli",
    # "subsample": 0.8,
    # "l2_leaf_reg": 3.0,
}

# -------------------------
# CPU 專屬參數
# -------------------------
CATBOOST_CPU_PARAMS = {
    "task_type": "CPU",

    # CPU 執行緒數量
    # 這是你想做控制實驗時最重要的參數之一
    "thread_count": 8,

    # 可選：限制 CPU 端某些記憶體使用情境
    # "used_ram_limit": "16gb",
}

# -------------------------
# GPU 專屬參數
# -------------------------
CATBOOST_GPU_PARAMS = {
    "task_type": "GPU",

    # 指定 GPU device，例如 "0" 或 "0:1"
    "devices": "0",

    # 允許 CatBoost 使用多少比例的 GPU 記憶體
    "gpu_ram_part": 0.90,

    # 可用的 pinned memory 大小
    "pinned_memory_size": "1gb",

    # 下列參數可依需要打開
    # "gpu_cat_features_storage": "GpuRam",   # 或 "CpuPinnedMemory"
    # "data_partition": "DocParallel",        # 多 GPU 比較有意義

    # 這個 thread_count 在 GPU 路徑下仍可能影響非核心部分
    "thread_count": 8,
}

# -------------------------
# 是否輸出 csv
# -------------------------
SAVE_RESULTS = False
RESULTS_CSV_PATH = "catboost_cpu_gpu_benchmark_raw.csv"
SUMMARY_CSV_PATH = "catboost_cpu_gpu_benchmark_summary.csv"


In [ ]:
# =========================
# Cell 3 — 補充說明：你現在的 benchmark 定義
# =========================

print(
    """Benchmark definition:
- Synthetic data generation time is excluded.
- CatBoost .fit() elapsed time is fully included.
- For GPU runs, this includes practical overheads on the GPU path
  (e.g. framework/device setup and data transfer-related costs visible in fit()).
- Input data is provided as pandas DataFrame for both CPU and GPU runs.
- CPU/GPU are compared under the same scenario, with the same generated dataset
  per (n_samples, n_features, run_id).
"""
)


In [ ]:
# =========================
# Cell 4 — Helper Functions
# =========================

@dataclass
class ScenarioConfig:
    n_samples: int
    n_features: int
    run_id: int


def make_dataframe_regression(
    n_samples: int,
    n_features: int,
    random_seed: int,
    test_size: float = 0.2,
    informative_ratio: float = 0.6,
    noise: float = 0.1,
    bias: float = 0.0,
    effective_rank: Optional[int] = None,
    tail_strength: float = 0.5,
    use_float32: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """
    產生一份 regression 資料，並轉成 pandas DataFrame / Series。

    注意：
    這個函式只負責產生資料，產生資料的時間不納入 benchmark。
    """
    n_informative = max(2, int(n_features * informative_ratio))

    X, y = make_regression(
        n_samples=n_samples,
        n_features=n_features,
        n_informative=n_informative,
        noise=noise,
        bias=bias,
        effective_rank=effective_rank,
        tail_strength=tail_strength,
        random_state=random_seed,
    )

    columns = [f"f_{i}" for i in range(n_features)]
    X_df = pd.DataFrame(X, columns=columns)
    y_sr = pd.Series(y, name="target")

    if use_float32:
        X_df = X_df.astype(np.float32)
        y_sr = y_sr.astype(np.float32)

    X_train, X_valid, y_train, y_valid = train_test_split(
        X_df,
        y_sr,
        test_size=test_size,
        random_state=random_seed,
    )

    return X_train, X_valid, y_train, y_valid


def build_model_params(device: str) -> Dict[str, Any]:
    """
    將通用參數與 CPU / GPU 專屬參數合併。
    """
    params = dict(CATBOOST_COMMON_PARAMS)

    if device.lower() == "cpu":
        params.update(CATBOOST_CPU_PARAMS)
    elif device.lower() == "gpu":
        params.update(CATBOOST_GPU_PARAMS)
    else:
        raise ValueError(f"Unsupported device: {device}")

    return params


def benchmark_single_fit(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_valid: pd.DataFrame,
    y_valid: pd.Series,
    device: str,
) -> Dict[str, Any]:
    """
    對單次 CatBoost fit 做 benchmark。

    設計重點：
    - 從 model 建立到 fit 完成的 elapsed time 都算進去
    - 對 GPU 來說，這樣就會把你在意的 setup / transfer / 路徑 overhead 算進去
    """
    params = build_model_params(device)

    start = time.perf_counter()
    model = CatBoostRegressor(**params)
    model.fit(
        X_train,
        y_train,
        eval_set=(X_valid, y_valid),
        use_best_model=False,
    )
    elapsed = time.perf_counter() - start

    result = {
        "device": device.lower(),
        "train_time_sec": elapsed,
        "best_iteration": getattr(model, "best_iteration_", None),
        "tree_count": getattr(model, "tree_count_", None),
        "model_params_json": json.dumps(params, ensure_ascii=False),
    }
    return result


In [ ]:
# =========================
# Cell 5 — 正式執行 Benchmark
# =========================
# 這一格會真的開始跑實驗
# 視你的 grid 大小、iterations、CPU / GPU 規模，可能需要一些時間

all_results: List[Dict[str, Any]] = []

scenario_grid = list(itertools.product(SAMPLE_LIST, FEATURE_LIST))
total_jobs = len(scenario_grid) * N_RUNS * 2
job_counter = 0

for n_samples, n_features in scenario_grid:
    for run_id in range(1, N_RUNS + 1):
        # 為了讓同一個 scenario/run 下的 CPU 與 GPU 用到同一份資料
        scenario_seed = RANDOM_SEED + run_id + (n_samples * 31) + (n_features * 17)

        # 產生資料（此段時間不納入 benchmark）
        X_train, X_valid, y_train, y_valid = make_dataframe_regression(
            n_samples=n_samples,
            n_features=n_features,
            random_seed=scenario_seed,
            test_size=TEST_SIZE,
            informative_ratio=N_INFORMATIVE_RATIO,
            noise=NOISE,
            bias=BIAS,
            effective_rank=EFFECTIVE_RANK,
            tail_strength=TAIL_STRENGTH,
            use_float32=USE_FLOAT32,
        )

        # ---- CPU benchmark ----
        cpu_result = benchmark_single_fit(
            X_train=X_train,
            y_train=y_train,
            X_valid=X_valid,
            y_valid=y_valid,
            device="cpu",
        )
        cpu_result.update({
            "n_samples": n_samples,
            "n_features": n_features,
            "run_id": run_id,
        })
        all_results.append(cpu_result)

        job_counter += 1
        print(f"[{job_counter}/{total_jobs}] CPU done | samples={n_samples}, features={n_features}, run={run_id}, time={cpu_result['train_time_sec']:.4f}s")

        # ---- GPU benchmark ----
        gpu_result = benchmark_single_fit(
            X_train=X_train,
            y_train=y_train,
            X_valid=X_valid,
            y_valid=y_valid,
            device="gpu",
        )
        gpu_result.update({
            "n_samples": n_samples,
            "n_features": n_features,
            "run_id": run_id,
        })
        all_results.append(gpu_result)

        job_counter += 1
        print(f"[{job_counter}/{total_jobs}] GPU done | samples={n_samples}, features={n_features}, run={run_id}, time={gpu_result['train_time_sec']:.4f}s")

df_raw = pd.DataFrame(all_results)

if SAVE_RESULTS:
    df_raw.to_csv(RESULTS_CSV_PATH, index=False)

print("\nRaw result preview:")
display(df_raw.head())
print(f"\nTotal rows: {len(df_raw)}")


In [ ]:
# =========================
# Cell 6 — 統計摘要
# =========================
# 這裡會把 raw results 整理成 scenario-level summary
# 並計算：
# - CPU / GPU mean time
# - CPU / GPU std time
# - speedup = CPU time / GPU time
# - time saved = CPU time - GPU time

def summarize_benchmark(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    required_cols = {
        "n_samples", "n_features", "run_id", "device", "train_time_sec"
    }
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    agg = (
        df.groupby(["n_samples", "n_features", "device"])["train_time_sec"]
        .agg(["mean", "std", "count", "min", "max"])
        .reset_index()
    )

    summary = agg.pivot(
        index=["n_samples", "n_features"],
        columns="device",
        values=["mean", "std", "count", "min", "max"]
    )

    summary.columns = [f"{dev}_{stat}" for stat, dev in summary.columns]
    summary = summary.reset_index()

    paired = (
        df.pivot_table(
            index=["n_samples", "n_features", "run_id"],
            columns="device",
            values="train_time_sec",
            aggfunc="mean"
        )
        .reset_index()
    )

    paired = paired.dropna(subset=["cpu", "gpu"]).copy()
    paired["speedup"] = paired["cpu"] / paired["gpu"]
    paired["time_saved_sec"] = paired["cpu"] - paired["gpu"]

    paired_stats = (
        paired.groupby(["n_samples", "n_features"])
        .agg(
            speedup_mean=("speedup", "mean"),
            speedup_std=("speedup", "std"),
            speedup_min=("speedup", "min"),
            speedup_max=("speedup", "max"),
            time_saved_mean=("time_saved_sec", "mean"),
            time_saved_std=("time_saved_sec", "std"),
            paired_runs=("speedup", "count"),
        )
        .reset_index()
    )

    summary_df = summary.merge(
        paired_stats,
        on=["n_samples", "n_features"],
        how="left",
    )

    return summary_df, paired


summary_df, paired_df = summarize_benchmark(df_raw)

if SAVE_RESULTS:
    summary_df.to_csv(SUMMARY_CSV_PATH, index=False)

print("Summary preview:")
display(summary_df.head())


In [ ]:
# =========================
# Cell 7 — 建立報告表
# =========================
# 這個表會比較適合直接看，或之後複製到報告中

def build_report_table(summary_df: pd.DataFrame) -> pd.DataFrame:
    out = summary_df.copy()

    out["cpu_time"] = out.apply(
        lambda r: f"{r['cpu_mean']:.4f} ± {r['cpu_std']:.4f}" if pd.notna(r["cpu_std"]) else f"{r['cpu_mean']:.4f}",
        axis=1
    )
    out["gpu_time"] = out.apply(
        lambda r: f"{r['gpu_mean']:.4f} ± {r['gpu_std']:.4f}" if pd.notna(r["gpu_std"]) else f"{r['gpu_mean']:.4f}",
        axis=1
    )
    out["speedup_fmt"] = out.apply(
        lambda r: f"{r['speedup_mean']:.3f} ± {r['speedup_std']:.3f}" if pd.notna(r["speedup_std"]) else f"{r['speedup_mean']:.3f}",
        axis=1
    )

    cols = [
        "n_samples",
        "n_features",
        "cpu_time",
        "gpu_time",
        "speedup_fmt",
        "time_saved_mean",
        "time_saved_std",
        "paired_runs",
    ]
    return out[cols].sort_values(["n_samples", "n_features"]).reset_index(drop=True)


report_table = build_report_table(summary_df)
display(report_table)


In [ ]:
# =========================
# Cell 8 — 快速結論摘要
# =========================
# 這一格會幫你快速看：
# 1. 哪個 scenario 的 GPU 加速比最高
# 2. 哪個 scenario 的絕對節省秒數最多
# 3. 哪些 scenario GPU 沒有優勢

best_speedup_row = summary_df.loc[summary_df["speedup_mean"].idxmax()].copy()
best_saved_row = summary_df.loc[summary_df["time_saved_mean"].idxmax()].copy()
worst_speedup_row = summary_df.loc[summary_df["speedup_mean"].idxmin()].copy()

print("=== 最佳 GPU 加速比 scenario ===")
print(best_speedup_row[["n_samples", "n_features", "speedup_mean", "speedup_std", "cpu_mean", "gpu_mean"]])

print("\n=== GPU 節省絕對秒數最多的 scenario ===")
print(best_saved_row[["n_samples", "n_features", "time_saved_mean", "time_saved_std", "cpu_mean", "gpu_mean"]])

print("\n=== GPU 相對最不划算的 scenario ===")
print(worst_speedup_row[["n_samples", "n_features", "speedup_mean", "speedup_std", "cpu_mean", "gpu_mean"]])

print("\n=== speedup > 1 的 scenario 數量 ===")
print((summary_df["speedup_mean"] > 1).sum(), "/", len(summary_df))


## 視覺化區

下面開始是圖表。  
如果你只想快速得到結果，至少建議先看：

1. `Speedup Heatmap`
2. `Absolute Time Saved Heatmap`
3. `Break-even Map`

這三張圖最適合拿來做結論。


In [ ]:
# =========================
# Cell 9 — Heatmap: Speedup
# =========================
# speedup = CPU time / GPU time
# > 1 表示 GPU 比 CPU 快
# < 1 表示 GPU 比 CPU 慢

heatmap_speedup = summary_df.pivot(
    index="n_samples",
    columns="n_features",
    values="speedup_mean"
).sort_index().sort_index(axis=1)

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(heatmap_speedup.values, aspect="auto")

ax.set_xticks(np.arange(len(heatmap_speedup.columns)))
ax.set_yticks(np.arange(len(heatmap_speedup.index)))
ax.set_xticklabels(heatmap_speedup.columns)
ax.set_yticklabels(heatmap_speedup.index)

ax.set_xlabel("Number of Features")
ax.set_ylabel("Number of Samples")
ax.set_title("GPU Speedup Heatmap (CPU time / GPU time)")

for i in range(heatmap_speedup.shape[0]):
    for j in range(heatmap_speedup.shape[1]):
        val = heatmap_speedup.iloc[i, j]
        if pd.notna(val):
            ax.text(j, i, f"{val:.2f}", ha="center", va="center")

plt.colorbar(im, ax=ax, label="Speedup")
plt.tight_layout()
plt.show()


In [ ]:
# =========================
# Cell 10 — Heatmap: Absolute Time Saved
# =========================
# 這張圖回答的是：
# GPU 平均幫你省下多少秒？
# time_saved = CPU time - GPU time

heatmap_saved = summary_df.pivot(
    index="n_samples",
    columns="n_features",
    values="time_saved_mean"
).sort_index().sort_index(axis=1)

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(heatmap_saved.values, aspect="auto")

ax.set_xticks(np.arange(len(heatmap_saved.columns)))
ax.set_yticks(np.arange(len(heatmap_saved.index)))
ax.set_xticklabels(heatmap_saved.columns)
ax.set_yticklabels(heatmap_saved.index)

ax.set_xlabel("Number of Features")
ax.set_ylabel("Number of Samples")
ax.set_title("Mean Time Saved by GPU (CPU time - GPU time, sec)")

for i in range(heatmap_saved.shape[0]):
    for j in range(heatmap_saved.shape[1]):
        val = heatmap_saved.iloc[i, j]
        if pd.notna(val):
            ax.text(j, i, f"{val:.2f}", ha="center", va="center")

plt.colorbar(im, ax=ax, label="Seconds Saved")
plt.tight_layout()
plt.show()


In [ ]:
# =========================
# Cell 11 — 固定 feature，看 CPU / GPU time vs samples
# =========================
# 這張圖可以看：
# 當 sample 數增加時，CPU 與 GPU 的訓練時間如何變化

selected_features = FEATURE_LIST[:min(3, len(FEATURE_LIST))]

for feat in selected_features:
    sub = summary_df[summary_df["n_features"] == feat].sort_values("n_samples")
    if sub.empty:
        continue

    plt.figure(figsize=(8, 5))
    plt.errorbar(
        sub["n_samples"],
        sub["cpu_mean"],
        yerr=sub["cpu_std"],
        marker="o",
        label="CPU"
    )
    plt.errorbar(
        sub["n_samples"],
        sub["gpu_mean"],
        yerr=sub["gpu_std"],
        marker="o",
        label="GPU"
    )

    if USE_LOG_X:
        plt.xscale("log")
    if USE_LOG_Y:
        plt.yscale("log")

    plt.xlabel("Number of Samples")
    plt.ylabel("Training Time (sec)")
    plt.title(f"Training Time vs Samples (n_features={feat})")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# =========================
# Cell 12 — 固定 sample，看 speedup vs features
# =========================
# 這張圖可以看：
# 當 feature 數增加時，GPU 加速比是否提升

selected_samples = SAMPLE_LIST[:min(3, len(SAMPLE_LIST))]

plt.figure(figsize=(9, 6))

for ns in selected_samples:
    sub = summary_df[summary_df["n_samples"] == ns].sort_values("n_features")
    if sub.empty:
        continue

    plt.errorbar(
        sub["n_features"],
        sub["speedup_mean"],
        yerr=sub["speedup_std"],
        marker="o",
        label=f"n_samples={ns}"
    )

plt.axhline(1.0, linestyle="--")
if USE_LOG_X:
    plt.xscale("log")

plt.xlabel("Number of Features")
plt.ylabel("Speedup (CPU time / GPU time)")
plt.title("GPU Speedup vs Number of Features")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# =========================
# Cell 13 — Break-even Map
# =========================
# 這張圖會把每個 scenario 分成三類：
# 1. GPU better
# 2. Similar
# 3. CPU better
#
# 分界標準你可以依需求修改

def classify_speedup(x: float) -> str:
    if pd.isna(x):
        return "NA"
    elif x > 1.2:
        return "GPU better"
    elif x < 0.9:
        return "CPU better"
    else:
        return "Similar"

break_even_df = summary_df.copy()
break_even_df["region"] = break_even_df["speedup_mean"].apply(classify_speedup)

pivot_region = break_even_df.pivot(
    index="n_samples",
    columns="n_features",
    values="region"
).sort_index().sort_index(axis=1)

pivot_speed = break_even_df.pivot(
    index="n_samples",
    columns="n_features",
    values="speedup_mean"
).sort_index().sort_index(axis=1)

fig, ax = plt.subplots(figsize=(10, 7))
ax.set_xticks(np.arange(len(pivot_region.columns)))
ax.set_yticks(np.arange(len(pivot_region.index)))
ax.set_xticklabels(pivot_region.columns)
ax.set_yticklabels(pivot_region.index)

ax.set_xlabel("Number of Features")
ax.set_ylabel("Number of Samples")
ax.set_title("Break-even Map")

for i in range(pivot_region.shape[0]):
    for j in range(pivot_region.shape[1]):
        region = pivot_region.iloc[i, j]
        speed = pivot_speed.iloc[i, j]

        if region == "GPU better":
            facecolor = "lightgreen"
        elif region == "CPU better":
            facecolor = "lightcoral"
        elif region == "Similar":
            facecolor = "khaki"
        else:
            facecolor = "lightgray"

        rect = plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=True, alpha=0.8, color=facecolor)
        ax.add_patch(rect)

        label = f"{region}\n{speed:.2f}" if pd.notna(speed) else region
        ax.text(j, i, label, ha="center", va="center", fontsize=9)

ax.set_xlim(-0.5, len(pivot_region.columns) - 0.5)
ax.set_ylim(len(pivot_region.index) - 0.5, -0.5)
plt.tight_layout()
plt.show()


## Notebook 使用建議

### 你最可能會先調整的東西

#### A. 想控制 CPU 強度
改這裡：

```python
CATBOOST_CPU_PARAMS = {
    "task_type": "CPU",
    "thread_count": 8,
}
```

#### B. 想控制 GPU 資源
改這裡：

```python
CATBOOST_GPU_PARAMS = {
    "task_type": "GPU",
    "devices": "0",
    "gpu_ram_part": 0.90,
    "pinned_memory_size": "1gb",
    "thread_count": 8,
}
```

#### C. 想讓 benchmark 更大或更小
改這裡：

```python
SAMPLE_LIST = [...]
FEATURE_LIST = [...]
N_RUNS = ...
```

#### D. 想讓模型更重
改這裡：

```python
CATBOOST_COMMON_PARAMS = {
    "iterations": 300,
    "depth": 6,
    "learning_rate": 0.1,
}
```

---

## 如何理解結果

### 1. Speedup Heatmap
- 若數值 **> 1**
  - 表示 GPU 較快
- 若數值 **< 1**
  - 表示 CPU 較快

### 2. Absolute Time Saved Heatmap
- 若數值很大
  - 表示 GPU 幫你省了很多秒
- 這比單純 speedup 更接近實務價值

### 3. Break-even Map
- `GPU better`
  - 代表在這個資料規模下 GPU 值得考慮
- `CPU better`
  - 代表在這個資料規模下，GPU overhead 可能還不划算
- `Similar`
  - 差異不大，可以依資源可用性決定

---

## 你可以怎麼寫最後結論
你通常可以整理成這種格式：

1. **小型資料**
   - GPU 可能沒有優勢，因為 setup / transfer overhead 占比太高

2. **中大型資料**
   - GPU 開始出現明顯優勢

3. **高樣本 / 高維度情境**
   - GPU 的 speedup 與 absolute time saving 往往更顯著

4. **實務判斷**
   - 若你的 pipeline 真正痛點是大規模 training latency，GPU 更可能值得
   - 若多數工作負載都偏小，CPU 可能更有效率也更穩定
